In [3]:
!pip install -q "google-genai>=2.12.1,<3" "google-auth==2.49.0"

import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google import genai
from google.genai import types
from google.colab import userdata

# ==========================================
# 1. BASE DE CONOCIMIENTO (CATÁLOGO DE CAFÉ)
# ==========================================
CATALOGO_CAFE = [
    {
        "id": "CAF-001",
        "nombre": "Huila Reserva Geisha",
        "origen": "Palestina, Huila",
        "altura": "1,950 msnm",
        "proceso": "Lavado",
        "tueste": "Claro",
        "notas": "Jazmín, bergamota, durazno blanco y miel de abeja",
        "acidez": "Alta y brillante",
        "cuerpo": "Sedoso y delicado",
        "metodo_recomendado": "V60, Chemex, Kalita",
        "precio_cop": 58000,
        "descripcion": "Café exótico de microlote con perfil floral intenso, ideal para métodos de filtrado."
    },
    {
        "id": "CAF-002",
        "nombre": "Sierra Nevada Orgánico",
        "origen": "Minca, Magdalena",
        "altura": "1,500 msnm",
        "proceso": "Lavado tradicional",
        "tueste": "Medio-Oscuro",
        "notas": "Chocolate oscuro, nuez tostada, panela y caramelo",
        "acidez": "Baja",
        "cuerpo": "Completo y cremoso",
        "metodo_recomendado": "Espresso, Prensa Francesa, Cafetera Italiana (Moka)",
        "precio_cop": 36000,
        "descripcion": "Cultivado bajo sombra. Perfil clásico, dulce y achocolatado con baja acidez."
    },
    {
        "id": "CAF-003",
        "nombre": "Quindío Pink Bourbon Honey",
        "origen": "Salento, Quindío",
        "altura": "1,800 msnm",
        "proceso": "Honey Amarillo",
        "tueste": "Medio",
        "notas": "Mandarina dulce, caña de azúcar, caramelo y frutos amarillos",
        "acidez": "Media-Alta jugosa",
        "cuerpo": "Redondo y balanceado",
        "metodo_recomendado": "AeroPress, V60, Cafetera de goteo",
        "precio_cop": 45000,
        "descripcion": "Equilibrio perfecto entre dulzor residual y acidez cítrica agradable."
    },
    {
        "id": "CAF-004",
        "nombre": "Nariño Volcánico Natural",
        "origen": "Buesaco, Nariño",
        "altura": "2,100 msnm",
        "proceso": "Natural (secado al sol en cereza)",
        "tueste": "Medio",
        "notas": "Frutos rojos maduros, fresa, vino tinto y cacao",
        "acidez": "Media vibrante",
        "cuerpo": "Vinoso y complejo",
        "metodo_recomendado": "Chemex, V60, Cold Brew",
        "precio_cop": 52000,
        "descripcion": "Gran altura que concentra azúcares, produciendo un perfil frutal intenso y fermentado."
    },
    {
        "id": "CAF-005",
        "nombre": "Antioquia Tradición Descafeinado",
        "origen": "Jardín, Antioquia",
        "altura": "1,700 msnm",
        "proceso": "Descafeinado natural con Acetato de Etilo (caña de azúcar)",
        "tueste": "Medio",
        "notas": "Panela, almendra dulce, manzana roja y chocolate con leche",
        "acidez": "Media-Baja suave",
        "cuerpo": "Medio",
        "metodo_recomendado": "Cualquier método (Goteo, Prensa Francesa, Espresso)",
        "precio_cop": 39000,
        "descripcion": "99.9% libre de cafeína sin químicos agresivos, ideal para tomar en la noche."
    },
    {
        "id": "CAF-006",
        "nombre": "Tolima Espresso Blend Intenso",
        "origen": "Planadas, Tolima",
        "altura": "1,650 msnm",
        "proceso": "Lavado + Natural",
        "tueste": "Oscuro",
        "notas": "Cacao amargo, especias, avellana tostada y melaza",
        "acidez": "Muy baja",
        "cuerpo": "Denso y persistente",
        "metodo_recomendado": "Máquina Espresso, Moka Italiana, Café con leche",
        "precio_cop": 34000,
        "descripcion": "Diseñado para cortar muy bien con leche y producir una crema densa."
    }
]

# ==========================================
# 2. MOTOR RAG + LLM (GEMINI)
# ==========================================
class AromaOrigenRAG:
    def __init__(self, api_key: str = None):
        self.df = pd.DataFrame(CATALOGO_CAFE)
        self.df["documento_indexado"] = self.df.apply(
            lambda r: (
                f"Producto: {r['nombre']} ({r['id']}). Origen: {r['origen']} a {r['altura']}. "
                f"Proceso: {r['proceso']}. Tueste: {r['tueste']}. "
                f"Notas de cata: {r['notas']}. Acidez: {r['acidez']}. Cuerpo: {r['cuerpo']}. "
                f"Métodos ideales: {r['metodo_recomendado']}. Precio: ${r['precio_cop']:,} COP. "
                f"Detalle: {r['descripcion']}"
            ),
            axis=1
        )

        self.vectorizer = TfidfVectorizer()
        self.tfidf_matrix = self.vectorizer.fit_transform(self.df["documento_indexado"])

        self.api_key = api_key
        self.client = genai.Client(api_key=self.api_key) if self.api_key else None

    def recuperar_contexto(self, consulta: str, top_k: int = 3) -> pd.DataFrame:
        query_vec = self.vectorizer.transform([consulta])
        similitudes = cosine_similarity(query_vec, self.tfidf_matrix).flatten()

        top_indices = np.argsort(similitudes)[::-1][:top_k]
        resultados = self.df.iloc[top_indices].copy()
        resultados["score_similitud"] = np.round(similitudes[top_indices], 4)
        return resultados

    def generar_respuesta(self, consulta_usuario: str, top_k: int = 3) -> dict:
        docs_recuperados = self.recuperar_contexto(consulta_usuario, top_k=top_k)

        contexto_texto = "\n".join(
            [f"- {row['documento_indexado']}" for _, row in docs_recuperados.iterrows()]
        )

        prompt_sistema = (
            "Eres 'Aroma', el barista virtual experto de la tienda de café de especialidad 'Aroma Origen'. "
            "Tu objetivo es asesorar al cliente y recomendarle el mejor café basándote ÚNICAMENTE en el contexto proporcionado. "
            "Sé cálido, persuasivo, conciso y menciona siempre el precio en COP y el método de preparación recomendado. "
            "Si el cliente pide algo que no está en el catálogo, indícalo amablemente y ofrece la alternativa más cercana."
        )

        prompt_usuario = f"""
CONTEXTO DEL CATÁLOGO RECUPERADO:
{contexto_texto}

CONSULTA DEL CLIENTE:
"{consulta_usuario}"

Responde con una recomendación clara, explicando por qué ese café encaja con lo que busca.
"""

        if self.client:
            try:
                response = self.client.models.generate_content(
                    model="gemini-2.5-flash",
                    contents=prompt_usuario,
                    config=types.GenerateContentConfig(
                        system_instruction=prompt_sistema,
                        temperature=0.4,
                    )
                )
                respuesta_final = response.text
                modo = "Gemini 2.5 Flash (LLM Activo)"
            except Exception as e:
                respuesta_final = self._respuesta_respaldo(docs_recuperados, error=str(e))
                modo = "Modo Respaldo (Error en API)"
        else:
            respuesta_final = self._respuesta_respaldo(docs_recuperados)
            modo = "Modo Respaldo Local (Sin API Key)"

        return {
            "respuesta": respuesta_final,
            "fuentes": docs_recuperados[["id", "nombre", "precio_cop", "score_similitud"]],
            "modo": modo
        }

    def _respuesta_respaldo(self, docs: pd.DataFrame, error: str = None) -> str:
        mejor = docs.iloc[0]
        aviso = f"\n*(Nota técnica: {error})*" if error else ""
        return (
            f"¡Hola! Basado en lo que buscas, te recomiendo nuestro **{mejor['nombre']}** "
            f"(Origen: {mejor['origen']}).\n\n"
            f"* **Perfil sensorial:** Notas a {mejor['notas']}, con acidez {mejor['acidez'].lower()}.\n"
            f"* **Preparación ideal:** {mejor['metodo_recomendado']}.\n"
            f"* **Precio:** ${mejor['precio_cop']:,} COP.\n\n"
            f"Es una excelente elección por su proceso {mejor['proceso'].lower()}.{aviso}"
        )

# ==========================================
# 3. EJECUCIÓN DE PRUEBA
# ==========================================
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = "" # Si no has puesto la key en Secrets, igual funcionará en modo respaldo

rag = AromaOrigenRAG(api_key=API_KEY)
pregunta = "Busco un café suave, sin mucha acidez, que sepa a chocolate para preparar en prensa francesa"

resultado = rag.generar_respuesta(pregunta)
print(f"--- MODO: {resultado['modo']} ---\n")
print(resultado["respuesta"])
print("\n--- DOCUMENTOS RECUPERADOS (RAG) ---")
display(resultado["fuentes"])

--- MODO: Modo Respaldo (Error en API) ---

¡Hola! Basado en lo que buscas, te recomiendo nuestro **Antioquia Tradición Descafeinado** (Origen: Jardín, Antioquia).

* **Perfil sensorial:** Notas a Panela, almendra dulce, manzana roja y chocolate con leche, con acidez media-baja suave.
* **Preparación ideal:** Cualquier método (Goteo, Prensa Francesa, Espresso).
* **Precio:** $39,000 COP.

Es una excelente elección por su proceso descafeinado natural con acetato de etilo (caña de azúcar).
*(Nota técnica: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}})*

--- DOCUMENTOS RECUPERADOS (RAG) ---


,id,nombre,precio_cop,score_similitud
4,CAF-005,Antioquia Tradición Descafeinado,39000,0.2711
3,CAF-004,Nariño Volcánico Natural,52000,0.1557
1,CAF-002,Sierra Nevada Orgánico,36000,0.1454


In [4]:
import gradio as gr
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# ==========================================
# 1. DATASET SINTÉTICO Y MODELO PREDICTIVO
# ==========================================
def crear_dataset_sintetico(n_muestras=1000):
    np.random.seed(42)
    tiempo_sitio_seg = np.random.randint(15, 600, n_muestras)
    paginas_vistas = np.random.randint(1, 15, n_muestras)
    mensajes_chat = np.random.randint(0, 10, n_muestras)
    presupuesto_cop = np.random.choice([35000, 45000, 55000, 70000], n_muestras)
    cliente_recurrente = np.random.binomial(1, 0.35, n_muestras)
    agrego_carrito = np.random.binomial(1, 0.40, n_muestras)

    # Lógica realista de probabilidad de conversión (compra = 1, no compra = 0)
    logit = (
        -3.2
        + 0.004 * tiempo_sitio_seg
        + 0.12 * paginas_vistas
        + 0.35 * mensajes_chat
        + 0.90 * cliente_recurrente
        + 2.10 * agrego_carrito
    )
    probabilidad = 1 / (1 + np.exp(-logit))
    compro = np.random.binomial(1, probabilidad)

    df_sintetico = pd.DataFrame({
        "tiempo_sitio_seg": tiempo_sitio_seg,
        "paginas_vistas": paginas_vistas,
        "mensajes_chat": mensajes_chat,
        "presupuesto_cop": presupuesto_cop,
        "cliente_recurrente": cliente_recurrente,
        "agrego_carrito": agrego_carrito,
        "compro": compro
    })
    return df_sintetico

# Generar datos y entrenar el modelo Random Forest
df_usuarios = crear_dataset_sintetico(1000)
X = df_usuarios.drop("compro", axis=1)
y = df_usuarios["compro"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
modelo_compra = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_compra.fit(X_train, y_train)

# Métricas para mostrar en el panel y usar en tu documento
accuracy = accuracy_score(y_test, modelo_compra.predict(X_test))
roc_auc = roc_auc_score(y_test, modelo_compra.predict_proba(X_test)[:, 1])

# ==========================================
# 2. CONECTAR EL ASISTENTE RAG + LLM
# ==========================================
# Aseguramos que use gemini-3.8-flash en la instancia de rag
import types as py_types

def generar_respuesta_v38(self, consulta_usuario: str, top_k: int = 3) -> dict:
    docs_recuperados = self.recuperar_contexto(consulta_usuario, top_k=top_k)
    contexto_texto = "\n".join(
        [f"- {row['documento_indexado']}" for _, row in docs_recuperados.iterrows()]
    )
    prompt_sistema = (
        "Eres 'Aroma', el barista virtual experto de la tienda de café de especialidad 'Aroma Origen'. "
        "Asesora al cliente y recomienda el mejor café basándote ÚNICAMENTE en el contexto proporcionado. "
        "Sé cálido, persuasivo, conciso y menciona siempre el precio en COP y el método de preparación recomendado."
    )
    prompt_usuario = f"CONTEXTO DEL CATÁLOGO:\n{contexto_texto}\n\nCLIENTE DICE: \"{consulta_usuario}\""

    if self.client:
        try:
            response = self.client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt_usuario,
                config=types.GenerateContentConfig(
                    system_instruction=prompt_sistema,
                    temperature=0.4,
                )
            )
            return {"respuesta": response.text, "fuentes": docs_recuperados}
        except Exception as e:
            return {"respuesta": self._respuesta_respaldo(docs_recuperados, str(e)), "fuentes": docs_recuperados}
    return {"respuesta": self._respuesta_respaldo(docs_recuperados), "fuentes": docs_recuperados}

rag.generar_respuesta = py_types.MethodType(generar_respuesta_v38, rag)

# ==========================================
# 3. FUNCIONES DE LA INTERFAZ WEB (GRADIO)
# ==========================================
def responder_chat(mensaje, historial):
    res = rag.generar_respuesta(mensaje)
    fuentes_str = " | ".join(
        [f"{r['nombre']} (Similitud: {r['score_similitud']})" for _, r in res["fuentes"].iterrows()]
    )
    respuesta_con_fuentes = f"{res['respuesta']}\n\n---\n🔍 *Fuentes RAG consultadas:* `{fuentes_str}`"
    return respuesta_con_fuentes

def predecir_intencion(tiempo, paginas, mensajes, presupuesto, recurrente, carrito):
    entrada = pd.DataFrame([{
        "tiempo_sitio_seg": tiempo,
        "paginas_vistas": paginas,
        "mensajes_chat": mensajes,
        "presupuesto_cop": presupuesto,
        "cliente_recurrente": 1 if recurrente else 0,
        "agrego_carrito": 1 if carrito else 0
    }])
    prob = modelo_compra.predict_proba(entrada)[0][1] * 100
    clase = "✅ ALTA PROBABILIDAD DE COMPRA (1)" if prob >= 50 else "⚠️ BAJA PROBABILIDAD DE COMPRA (0)"
    accion = (
        "💡 **Acción recomendada:** Ofrecer envío gratis o descuento del 10% en el checkout para cerrar la venta."
        if prob >= 50 else
        "💡 **Acción recomendada:** Activar recomendación proactiva del asistente Aroma para descubrir sus gustos."
    )
    return f"### {clase}\n**Probabilidad estimada:** `{prob:.1f}%`\n\n{accion}"

# ==========================================
# 4. DISEÑO VISUAL DE LA APP WEB
# ==========================================
with gr.Blocks(theme=gr.themes.Soft(primary_hue="amber", secondary_hue="stone")) as demo:
    gr.Markdown(
        """
        # ☕ Aroma Origen — Asistente Virtual de E-Commerce con IA
        **Arquitectura:** RAG (TF-IDF + Coseno) + LLM (`Gemini 3.8 Flash`) + Predicción Binaria de Compra (`Random Forest`)
        """
    )

    with gr.Tabs():
        # PESTAÑA 1: ASISTENTE VIRTUAL PARA EL CLIENTE
        with gr.Tab("💬 Asistente Virtual (Cliente)"):
            gr.ChatInterface(
                fn=responder_chat,
                examples=[
                    "Busco un café suave con notas a chocolate para prensa francesa",
                    "Quiero un café floral y exótico para preparar en V60",
                    "¿Tienen algún café sin cafeína para tomar en la noche?",
                    "Necesito un café intenso para máquina espresso con leche"
                ],
                title="Habla con Aroma, tu Barista Virtual",
                description="Pregúntame por sabores, métodos de preparación, orígenes o presupuesto."
            )

        # PESTAÑA 2: CATÁLOGO DE PRODUCTOS (BASE DE CONOCIMIENTO)
        with gr.Tab("📦 Catálogo de Especialidad"):
            gr.Markdown("### Base de Conocimiento Indexada en el Sistema RAG")
            gr.Dataframe(
                value=rag.df[["id", "nombre", "origen", "proceso", "tueste", "notas", "metodo_recomendado", "precio_cop"]],
                interactive=False
            )

        # PESTAÑA 3: MODELO PREDICTIVO DE COMPRA (ANALÍTICA)
        with gr.Tab("📊 Predicción de Compra (ML)"):
            gr.Markdown(
                f"""
                ### Simulador de Predicción Binaria de Conversión
                * **Dataset Sintético:** `1,000` sesiones simuladas de e-commerce.
                * **Exactitud (Accuracy):** `{accuracy*100:.2f}%` | **ROC-AUC:** `{roc_auc:.3f}`
                """
            )
            with gr.Row():
                with gr.Column():
                    in_tiempo = gr.Slider(15, 600, value=180, label="Tiempo en el sitio (segundos)")
                    in_paginas = gr.Slider(1, 15, value=5, step=1, label="Páginas de producto vistas")
                    in_mensajes = gr.Slider(0, 10, value=3, step=1, label="Interacciones con el Asistente IA")
                    in_presupuesto = gr.Dropdown([35000, 45000, 55000, 70000], value=45000, label="Presupuesto estimado (COP)")
                    in_recurrente = gr.Checkbox(value=True, label="¿Es cliente recurrente?")
                    in_carrito = gr.Checkbox(value=False, label="¿Agregó producto al carrito?")
                    btn_predecir = gr.Button("Calcular Probabilidad de Compra", variant="primary")
                with gr.Column():
                    salida_prediccion = gr.Markdown("### Ajusta los parámetros y haz clic en calcular.")

            btn_predecir.click(
                fn=predecir_intencion,
                inputs=[in_tiempo, in_paginas, in_mensajes, in_presupuesto, in_recurrente, in_carrito],
                outputs=salida_prediccion
            )

# Lanzar aplicación con URL pública
demo.launch(share=True)

/tmp/ipykernel_37440/2558122664.py:123: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="amber", secondary_hue="stone")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b0f6a97d3e8bd4d8a9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
